# Read and clean social media data

This notebook loads the local `synthetic_social_media_data.csv`, audits and cleans its records, saves a cleaned copy, and summarizes engagement patterns. Run from any working directory; file paths are resolved relative to this notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path.cwd() / "synthetic_social_media_data.csv"
if not DATA_PATH.exists():
    # Also support running the notebook when the kernel starts elsewhere.
    candidates = [Path("D:/predictive_analysis/synthetic_social_media_data.csv"), Path("./synthetic_social_media_data.csv")]
    DATA_PATH = next((path for path in candidates if path.exists()), DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError("Place synthetic_social_media_data.csv beside this notebook or update DATA_PATH.")

CLEAN_PATH = DATA_PATH.with_name("cleaned_social_media_data.csv")
raw_df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(raw_df):,} rows and {len(raw_df.columns)} columns from {DATA_PATH}")
raw_df.head()

## Audit the source data

Review schema, missing values, duplicate rows, numeric ranges, and category values before cleaning.

In [ ]:
required_columns = ["Platform", "Views", "Likes", "Shares", "Comments", "Engagement_Level"]
missing_columns = sorted(set(required_columns) - set(raw_df.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

audit = pd.DataFrame({
    "dtype": raw_df[required_columns].dtypes.astype(str),
    "missing": raw_df[required_columns].isna().sum(),
    "unique_values": raw_df[required_columns].nunique(dropna=True),
})
print("Shape:", raw_df.shape)
print("Exact duplicate rows:", int(raw_df.duplicated().sum()))
display(audit)
display(raw_df[["Views", "Likes", "Shares", "Comments"]].describe().T)
print("Negative interaction counts:")
display((raw_df[["Views", "Likes", "Shares", "Comments"]].lt(0)).sum().rename("negative_rows").to_frame())
print("Platform values:", sorted(raw_df["Platform"].dropna().astype(str).str.strip().unique()))
print("Engagement labels:", sorted(raw_df["Engagement_Level"].dropna().astype(str).str.strip().unique()))

## Clean records

Text fields are trimmed, metric columns are converted to numeric, invalid or missing required values are removed, exact duplicate records are dropped, negative counts are set to zero, and known categories are normalized. The cleaning log records how many rows were affected.

In [ ]:
df = raw_df.copy()
start_rows = len(df)
text_columns = ["Platform", "Engagement_Level"]
for column in text_columns:
    df[column] = df[column].astype("string").str.strip()
metric_columns = ["Views", "Likes", "Shares", "Comments"]
for column in metric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["Platform"] = df["Platform"].str.casefold().map({"instagram": "Instagram", "youtube": "YouTube"})
df["Engagement_Level"] = df["Engagement_Level"].str.title()
valid_platforms = {"Instagram", "YouTube"}
valid_levels = {"Low", "Medium", "High"}
invalid_categories = (~df["Platform"].isin(valid_platforms)) | (~df["Engagement_Level"].isin(valid_levels))
missing_required = df[required_columns].isna().any(axis=1)
rows_dropped_invalid = int((invalid_categories | missing_required).sum())
df = df.loc[~(invalid_categories | missing_required)].copy()
rows_dropped_duplicates = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
negative_counts = df[metric_columns].lt(0).sum()
negative_values_fixed = int(negative_counts.sum())
df[metric_columns] = df[metric_columns].clip(lower=0)
df[metric_columns] = df[metric_columns].round().astype("int64")

cleaning_log = pd.Series({
    "input_rows": start_rows,
    "rows_dropped_missing_or_invalid": rows_dropped_invalid,
    "duplicate_rows_removed": rows_dropped_duplicates,
    "negative_metric_values_clipped_to_zero": negative_values_fixed,
    "output_rows": len(df),
}, name="count")
display(cleaning_log.to_frame())
display(df.head())
print("Remaining missing values:", int(df[required_columns].isna().sum().sum()))
print("Remaining negative metrics:", int(df[metric_columns].lt(0).sum().sum()))
df.to_csv(CLEAN_PATH, index=False)
print(f"Cleaned data saved to: {CLEAN_PATH}")

## Explore engagement

Compare the class balance and average reach and interaction counts across engagement labels and platforms.

In [ ]:
engagement_counts = df["Engagement_Level"].value_counts().reindex(["Low", "Medium", "High"], fill_value=0)
engagement_share = (engagement_counts / len(df) * 100).round(2)
class_balance = pd.DataFrame({"posts": engagement_counts, "share_percent": engagement_share})
display(class_balance)

platform_summary = df.groupby("Platform").agg(
    posts=("Platform", "size"),
    avg_views=("Views", "mean"),
    median_views=("Views", "median"),
    avg_likes=("Likes", "mean"),
    avg_shares=("Shares", "mean"),
    avg_comments=("Comments", "mean"),
).round(2)
display(platform_summary)

engagement_summary = df.groupby("Engagement_Level")[metric_columns].agg(["count", "mean", "median"]).round(2)
display(engagement_summary)

## Interaction rates and relationships

Rates use views as the denominator; rows with zero views receive a rate of zero. Correlations describe associations in this dataset and do not establish causation.

In [ ]:
for metric in ["Likes", "Shares", "Comments"]:
    df[f"{metric}_per_view"] = np.divide(
        df[metric].to_numpy(dtype=float),
        df["Views"].to_numpy(dtype=float),
        out=np.zeros(len(df), dtype=float),
        where=df["Views"].to_numpy(dtype=float) > 0,
    )
    df[f"{metric}_per_view"] *= 100
rate_columns = ["Likes_per_view", "Shares_per_view", "Comments_per_view"]
print("Mean interaction rates by platform (% of views):")
display(df.groupby("Platform")[rate_columns].mean().round(3))
print("Numeric feature correlations:")
display(df[metric_columns].corr().round(3))
print("Engagement label by platform (% within platform):")
display(pd.crosstab(df["Platform"], df["Engagement_Level"], normalize="index").mul(100).round(2))

## Findings and saved output

Use the tables above to identify which platform has higher average reach and how interaction metrics vary across engagement labels. The source labels are retained as provided; this analysis does not retrain or validate a predictive model.

In [ ]:
print(f"Clean dataset: {len(df):,} rows, {len(df.columns)} columns (including derived rate features).")
print(f"Cleaned source table saved to {CLEAN_PATH}")
print("Most common engagement level:", engagement_counts.idxmax())
print("Platform with the highest average views:", platform_summary["avg_views"].idxmax())